# Sales channels — Supplementary exploration

Author-origin quantitative analysis retained. See the English consolidated case study for validated interpretation and executive storytelling. The original explorations were produced before the portfolio restructuring and may contain Italian in analytical print messages.

## Setup — caricamento e controlli

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 110, 'axes.titleweight':'semibold', 'axes.labelsize':10, 'font.size':10})
pd.set_option('display.max_colwidth', 75)
pd.set_option('display.max_rows', 80)

candidates = [
    Path.cwd() / 'output' / 'adidas_sales_features.csv',
    Path.cwd().parent / 'output' / 'adidas_sales_features.csv',
    Path.cwd().parent.parent / 'output' / 'adidas_sales_features.csv',
    Path.cwd() / 'adidas_sales_features.csv',
    Path.cwd().parent / 'adidas_sales_features.csv',
]
file_input = next((p for p in candidates if p.is_file()), None)
if file_input is None:
    raise FileNotFoundError('Manca output/adidas_sales_features.csv. Esegui i notebook 01 e 02 del progetto, poi avvia questo notebook dalla root o dalla cartella notebooks/.')
df = pd.read_csv(file_input, parse_dates=['invoice_date'], dtype={'retailer_id': 'string'})
expected = {'retailer','invoice_date','region','state','city','product','price_per_unit',
            'units_sold','total_sales','operating_profit','operating_margin','sales_method',
            'year','month_number','month_name_it','year_month','weekday_number',
            'weekday_name_it','is_weekend','season_it','product_gender','product_category'}
assert expected.issubset(df.columns), f'Colonne mancanti: {expected-set(df.columns)}'
assert not df['invoice_date'].isna().any(), 'Date mancanti'
assert np.allclose(df['total_sales'], df['price_per_unit']*df['units_sold'],atol=.011), 'Ricavi non coerenti'
assert np.allclose(df['operating_profit'],df['total_sales']*df['operating_margin'],atol=.011), 'Profitto non coerente'

EURO_MONTH=['Gennaio','Febbraio','Marzo','Aprile','Maggio','Giugno','Luglio','Agosto','Settembre','Ottobre','Novembre','Dicembre']
DAYS=['Lunedì','Martedì','Mercoledì','Giovedì','Venerdì','Sabato','Domenica']
SEASONS=['Inverno','Primavera','Estate','Autunno']

def units(x): return f'{x:,.0f}'.replace(',', '.')
def usd(x): return f'${x:,.2f}' if abs(x)<1000 else f'${x:,.0f}'
def percent(x): return 'N/D' if pd.isna(x) else f'{x:,.2f}%'
def safe(a,b): return np.nan if b==0 or pd.isna(b) else a/b
def avg(x): return f'{x:,.2f}'
def note(message): display(Markdown('**Risultato:** '+str(message)))
def caution(message): display(Markdown('**Limite interpretativo:** '+str(message)))
def data_table(frame, rows=80):
    display(frame.head(rows).reset_index(drop=True).round(2))
    if len(frame)>rows:
        display(Markdown(f'_Visualizzate {rows} righe su {len(frame)}. La tabella completa rimane disponibile nella variabile Python._'))

def summary(dims, data=None):
    frame=df if data is None else data
    group=frame.groupby(dims,dropna=False,observed=True)
    result=group.agg(units=('units_sold','sum'),revenue=('total_sales','sum'),
                     profit=('operating_profit','sum'),rows=('units_sold','size'),
                     price_mean=('price_per_unit','mean'),days=('invoice_date','nunique')).reset_index()
    result['margin_pct']=np.where(result.revenue!=0,100*result.profit/result.revenue,np.nan)
    result['realized_price']=np.where(result.units!=0,result.revenue/result.units,np.nan)
    result['profit_unit']=np.where(result.units!=0,result.profit/result.units,np.nan)
    return result

def dominance(frame,dims,category,measure='units'):
    x=frame.sort_values(dims+[measure,category],ascending=[True]*len(dims)+[False,True])
    first=x.drop_duplicates(subset=dims,keep='first').copy()
    top=frame.groupby(dims)[measure].transform('max')
    tied=frame[frame[measure].eq(top)].groupby(dims,dropna=False)[category].nunique().reset_index(name='ex_aequo')
    return first.merge(tied,on=dims,how='left')

def bar(frame,label,value,title,top=12,ascending=False,ax=None):
    d=frame.sort_values(value,ascending=ascending).head(top).copy()
    if ax is None: fig,ax=plt.subplots(figsize=(11,max(4,0.40*len(d)+1.5)))
    else: fig=ax.figure
    d=d.iloc[::-1]
    ax.barh(d[label].astype(str),d[value]);ax.set_title(title);ax.set_xlabel(value.replace('_',' ').title());ax.grid(axis='x',alpha=.25)
    ax.tick_params(axis='y',labelsize=9)
    fig.tight_layout()
    return ax

def twobars(frame,label,metrics,titles,top=12):
    fig,axes=plt.subplots(1,2,figsize=(15,max(4,0.43*min(top,len(frame))+1)))
    for ax,m,t in zip(axes,metrics,titles):bar(frame,label,m,t,top=top,ax=ax)
    plt.show()

def stack(data,index,columns,measure,title,top=14,shares=True):
    g=data.pivot_table(index=index,columns=columns,values=measure,aggfunc='sum',fill_value=0)
    if g.empty:
        note('Non sono presenti dati per questo confronto.');return
    g=g.loc[g.sum(axis=1).sort_values(ascending=False).head(top).index]
    if shares: g=g.div(g.sum(axis=1).replace(0,np.nan),axis=0).fillna(0)*100
    ax=g.plot(kind='bar',stacked=True,figsize=(max(10, min(16,len(g)*0.7+5)),5.5),width=.80)
    ax.set_title(title);ax.set_ylabel('Quota sul volume (%)' if shares else measure)
    ax.set_xlabel('');ax.legend(title=columns,bbox_to_anchor=(1.01,1),loc='upper left')
    plt.xticks(rotation=40,ha='right');plt.tight_layout();plt.show()

def heat(data,index,columns,value,title,top=25,normalize=False):
    piv=data.pivot_table(index=index,columns=columns,values=value,aggfunc='sum',fill_value=0)
    if piv.empty: return
    piv=piv.loc[piv.sum(axis=1).sort_values(ascending=False).head(top).index]
    if normalize: piv=piv.div(piv.sum(axis=1).replace(0,np.nan),axis=0).fillna(0)*100
    fig,ax=plt.subplots(figsize=(max(8,1.15*len(piv.columns)+4),max(4,.36*len(piv)+1.8)))
    im=ax.imshow(piv.to_numpy(),aspect='auto',cmap='Blues')
    ax.set_xticks(np.arange(len(piv.columns)),piv.columns.astype(str),rotation=35,ha='right')
    ax.set_yticks(np.arange(len(piv.index)),piv.index.astype(str));ax.set_title(title)
    fig.colorbar(im,ax=ax,shrink=.80,label='Quota (%)' if normalize else value)
    fig.tight_layout();plt.show()

def lineplot(frame,x,y,title,color_key=None):
    fig,ax=plt.subplots(figsize=(12,5))
    if color_key is None:
        t=frame.sort_values(x);ax.plot(t[x].astype(str),t[y],marker='o',markersize=3)
    else:
        for name,piece in frame.groupby(color_key,observed=True):
            t=piece.sort_values(x);ax.plot(t[x].astype(str),t[y],marker='o',markersize=3,label=str(name))
        ax.legend(title=color_key)
    ax.set_title(title);ax.set_ylabel(y);ax.tick_params(axis='x',rotation=70,labelsize=8)
    ax.grid(alpha=.25);fig.tight_layout();plt.show()

note(f'Caricate {units(len(df))} righe • periodo {df.invoice_date.min():%Y-%m-%d} — {df.invoice_date.max():%Y-%m-%d} • nessun CSV verrà esportato.')

## Which selling method leads in volume and revenue?

In [ ]:
t=summary(['sales_method']).sort_values('units',ascending=False);r=t.iloc[0]
note(f"Il canale leader per unità è {r.sales_method}: {units(r.units)} unità. Il primato per ricavi va verificato separatamente.")
twobars(t,'sales_method',['units','revenue'],['Unità per canale','Ricavi per canale (USD)'])
data_table(t)

## Which selling method leads in profit and margin?

In [ ]:
t=summary(['sales_method']);a=t.loc[t.profit.idxmax()];b=t.loc[t.margin_pct.idxmax()]
note(f"Profitto più alto: {a.sales_method} ({usd(a.profit)}). Margine più alto: {b.sales_method} ({percent(b.margin_pct)}). Non sono lo stesso indicatore.")
twobars(t,'sales_method',['profit','margin_pct'],['Profitto operativo (USD)','Margine aggregato (%)'])
data_table(t)

## Which selling method dominates by territory?

In [ ]:
locs=[['region'],['region','state'],['region','state','city']]
tables=[summary(x+['sales_method']) for x in locs];winners=[dominance(t,l,'sales_method') for t,l in zip(tables,locs)]
note('Per ogni regione, stato e città è scelto il canale con più unità (con eventuale ex aequo riportato).')
heat(tables[0],'region','sales_method','units','Quota canali entro la regione (%)',normalize=True)
for l,w in zip(locs,winners):data_table(w[l+['sales_method','units','ex_aequo']])

## How does channel composition vary by target and category?

In [ ]:
a=summary(['product_gender','sales_method']);b=summary(['product_category','sales_method'])
note('Quote per unità, calcolate separatamente entro ogni target e ogni macro-categoria di prodotto.')
fig,axes=plt.subplots(1,2,figsize=(15,5))
for ax,t,index,name in [(axes[0],a,'product_gender','Target'),(axes[1],b,'product_category','Categoria')]:
    p=t.pivot(index=index,columns='sales_method',values='units').fillna(0)
    p=p.div(p.sum(axis=1),axis=0)*100;p.plot(kind='bar',stacked=True,ax=ax)
    ax.set_title('Canali per '+name);ax.set_ylabel('% unità');ax.tick_params(axis='x',rotation=0);ax.legend(title='Canale',fontsize=8)
plt.tight_layout();plt.show()
for t,col in [(a,'product_gender'),(b,'product_category')]:
    t['share_pct']=100*t.units/t.groupby(col).units.transform('sum');data_table(t)

## Which selling method dominates each retailer?

In [ ]:
t=summary(['retailer','sales_method']);w=dominance(t,['retailer'],'sales_method')
note('Canale con più unità per ogni retailer; nei risultati compare anche ricavo, profitto e margine per il canale vincente.')
stack(t,'retailer','sales_method','units','Mix canali per retailer (%)')
data_table(w[['retailer','sales_method','units','revenue','profit','margin_pct','ex_aequo']]);data_table(t)

## How does channel mix vary over time?

In [ ]:
dims=['year','month_number','weekday_number','is_weekend','season_it']
labels=['Anno','Mese','Giorno settimana','Weekend','Stagione']
fig,axes=plt.subplots(3,2,figsize=(17,15));axes=axes.ravel()
for k,(dim,label) in enumerate(zip(dims,labels)):
    t=summary([dim,'sales_method'])
    piv=t.pivot(index=dim,columns='sales_method',values='units').fillna(0)
    piv=piv.div(piv.sum(axis=1),axis=0)*100
    if dim=='season_it':piv=piv.reindex(SEASONS)
    if dim=='is_weekend':piv.index=piv.index.map({False:'Feriale',True:'Weekend'})
    if dim=='month_number':piv.index=[EURO_MONTH[int(x)-1] for x in piv.index]
    if dim=='weekday_number':piv.index=[DAYS[int(x)] for x in piv.index]
    piv.plot(kind='bar',stacked=True,ax=axes[k],legend=False);axes[k].set_title('Mix per '+label);axes[k].set_ylabel('% unità')
    axes[k].tick_params(axis='x',rotation=45)
axes[5].axis('off');axes[4].legend(title='Canale',bbox_to_anchor=(1.03,1),loc='upper left')
fig.tight_layout()
note('Sono mostrati i mix relativi ai cinque livelli temporali; per i mesi la quota aggrega entrambi gli anni, che hanno coperture diverse.')
plt.show()
for dim in dims:
    t=summary([dim,'sales_method']);t['share_pct']=100*t.units/t.groupby(dim).units.transform('sum');data_table(t)